In [143]:
import pandas as pd
import geopandas as gp
from collections import Counter
import pber_functions_v1 as pdv
import os

## Load Files

In [99]:
la_csv = pd.read_csv("./la_2024_gen_prec/la_2024_gen_prec.csv")

In [100]:
# Shapes
la_shapes = gp.read_file("./raw-from-source/2024 Precinct Shapefiles (12-31-2024)/_2024 Precinct Shapefiles (12-31-2024).shp")

# Election results
la_elections = pd.read_csv("./la_2024_gen_prec_turnout/la_2024_gen_prec_turnout.csv")
la_elections["join_id"] = la_elections["COUNTYFP"].astype(str) + la_elections["Precinct"].astype(str)

In [101]:
la_elections = la_elections[['UNIQUE_ID', 'COUNTYFP', 'Parish', 'Precinct', 'G24A1NO', 'G24A1YES',
       'G24PREDHAR', 'G24PRELOLI', 'G24PREOCRU', 'G24PREOFRU', 'G24PREOKEN',
       'G24PREOPRE', 'G24PREOSON', 'G24PREOSTE', 'G24PREOTER', 'G24PREOWES',
       'G24PRERTRU', 'GCON01DMAN', 'GCON01NHYE', 'GCON01RARR', 'GCON01RSCA',
       'GCON01RSHA', 'GCON02DCAR', 'GCON02DDAV', 'GCON02RGRA', 'GCON02RLYN',
       'GCON02RPER', 'GCON03DGON', 'GCON03DSUM', 'GCON03RHIG', 'GCON03RJOH',
       'GCON04RJOH', 'GCON04RMOR', 'GCON05DVAL', 'GCON05RLET', 'GCON05RMEN',
       'GCON06DAND', 'GCON06DFIE', 'GCON06DJON', 'GCON06DWIL', 'GCON06RGUI','join_id']]

In [102]:
# la_shapes.to_csv("./la2024_shp.csv", index = False)
# la_elections.to_csv("./la2024_results.csv", index = False)

In [103]:
races = [i for i in list(la_elections.columns) if i[0]=="G"]

In [104]:
prec_values = pd.read_csv("./raw-from-source/prec_joining_table.csv")

In [105]:
prec_values[~prec_values["UNIQUE_ID"].isin(la_elections["UNIQUE_ID"])]

,GEOID20,DISSOLVE_ALPHA,UNIQUE_ID
1294,2204722B,2204722B,Iberville-:-00 22B
1636,220610012-3,220610012-3,Lincoln-:-12 03
2629,220955-5,220955-5,St. John The Baptist-:-05 05
3314,221252B,221252B,West Feliciana-:-00 02B
3516,22115005-2A,22115005-2A,Vernon-:-05 02A


In [147]:
prec_values[~prec_values["GEOID20"].isin(la_shapes["GEOID20"])]["UNIQUE_ID"].apply(lambda x: x.split("-:-")[0]).unique()

array(['Ascension', 'Assumption', 'Bossier', 'East Baton Rouge',
       'Lafourche', 'Rapides', 'St. Charles', 'Terrebonne'], dtype=object)

In [107]:
prec_values[prec_values["GEOID20"]!=prec_values["DISSOLVE_ALPHA"]][["DISSOLVE_ALPHA","UNIQUE_ID"]]

,DISSOLVE_ALPHA,UNIQUE_ID
66,22005000001,Ascension-:-00 01 B
74,22005000008,Ascension-:-00 08 B
78,22005000011,Ascension-:-00 11 B
94,22005000026,Ascension-:-00 26 B
156,220070006-1,Assumption-:-06 01 A
...,...,...
2565,220890007-2,St. Charles-:-07 2 A
2567,220890007-3,St. Charles-:-07 3 A
2569,220890007-4,St. Charles-:-07 4 A
3082,22109000007,Terrebonne-:-00 007 L


In [108]:
elec_merge_data = pd.merge(la_elections, prec_values, left_on = "UNIQUE_ID", right_on = "UNIQUE_ID", how = "outer")

In [109]:
elec_merge_data = elec_merge_data.fillna(0)

In [110]:
elec_merge_data.columns

Index(['UNIQUE_ID', 'COUNTYFP', 'Parish', 'Precinct', 'G24A1NO', 'G24A1YES',
       'G24PREDHAR', 'G24PRELOLI', 'G24PREOCRU', 'G24PREOFRU', 'G24PREOKEN',
       'G24PREOPRE', 'G24PREOSON', 'G24PREOSTE', 'G24PREOTER', 'G24PREOWES',
       'G24PRERTRU', 'GCON01DMAN', 'GCON01NHYE', 'GCON01RARR', 'GCON01RSCA',
       'GCON01RSHA', 'GCON02DCAR', 'GCON02DDAV', 'GCON02RGRA', 'GCON02RLYN',
       'GCON02RPER', 'GCON03DGON', 'GCON03DSUM', 'GCON03RHIG', 'GCON03RJOH',
       'GCON04RJOH', 'GCON04RMOR', 'GCON05DVAL', 'GCON05RLET', 'GCON05RMEN',
       'GCON06DAND', 'GCON06DFIE', 'GCON06DJON', 'GCON06DWIL', 'GCON06RGUI',
       'join_id', 'GEOID20', 'DISSOLVE_ALPHA'],
      dtype='object')

In [111]:
data_for_join = elec_merge_data.groupby("DISSOLVE_ALPHA", as_index = False).sum(numeric_only=True)

In [112]:
data_for_join.drop(["COUNTYFP"], axis = 1, inplace = True)

In [113]:
final_join = gp.GeoDataFrame(pd.merge(la_shapes, data_for_join, left_on = "GEOID20", right_on = "DISSOLVE_ALPHA", how = "outer", indicator = True))

In [114]:
final_join["_merge"].value_counts()

_merge
both          3651
left_only        5
right_only       0
Name: count, dtype: int64

In [115]:
final_join[final_join["_merge"]=="left_only"].sort_values("COUNTYFP20")

,ID,AREA,OBJECTID,STATEFP20,COUNTYFP20,VTDST20,GEOID20,VTDI20,NAME20,NAMELSAD20,...,GCON04RMOR,GCON05DVAL,GCON05RLET,GCON05RMEN,GCON06DAND,GCON06DFIE,GCON06DJON,GCON06DWIL,GCON06RGUI,_merge
1516,2677,94.970009,3249,22,051,ZZZZZZ,22051ZZZZZZ,P,Voting Districts Not Defined,Voting Districts Not Defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2365,2718,142.064377,1962,22,071,ZZZZZZ,22071ZZZZZZ,P,Voting Districts Not Defined,Voting Districts Not Defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2734,1669,40.213352,2660,22,089,ZZZZZZ,22089ZZZZZZ,P,Voting Districts Not Defined,Voting Districts Not Defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2802,1380,160.514709,1387,22,095,ZZZZZZ,22095ZZZZZZ,P,Voting Districts Not Defined,Voting Districts Not Defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3227,952,32.063110,926,22,105,ZZZZZZ,22105ZZZZZZ,P,Voting Districts Not Defined,Voting Districts Not Defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [116]:
final_join.drop("_merge", axis = 1, inplace = True)

In [117]:
final_join = final_join.fillna(0)

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_9433/2443853812.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_join = final_join.fillna(0)


In [118]:
final_join.drop(["DISSOLVE_ALPHA"], axis = 1, inplace = True)

In [119]:
final_join.rename(columns = {'COUNTYFP20':'COUNTYFP'}, inplace = True)
counties = pd.read_csv("./raw-from-source/FIPS/US_FIPS_Codes.csv", dtype =str)
la_counties = counties[counties["State"]=="Louisiana"]
la_counties_dict = dict(zip(la_counties["FIPS County"], la_counties["County Name"]))
final_join["Parish"] = final_join["COUNTYFP"].map(la_counties_dict).fillna("MISSING")

In [125]:
final_join[["Parish","NAME20"]].value_counts(dropna = False)

Parish    NAME20
Acadia    1-1       1
Ouachita  62        1
          64        1
          65        1
          65A       1
                   ..
Jackson   16        1
          17        1
          17A       1
          18        1
Winn      7-7       1
Name: count, Length: 3656, dtype: int64

In [126]:
final_join["UNIQUE_ID"] = final_join["Parish"] + "-:-" + final_join["NAME20"]

In [127]:
#final_join["UNIQUE_ID"] = final_join["GEOID20"]

In [128]:
def is_split_precinct(district_assignment_list):
    c = Counter([x[0] for x in district_assignment_list])
    greater_than_one = {x:[y[1] for y in district_assignment_list if y[0]==x] for x, count in c.items() if count > 1}
    if len(greater_than_one)==0:
        return 0
    else:
        return greater_than_one
    
def get_level_dist(column_name):
    if column_name[0:3] == "GSL" or column_name[0:3] == "PSL":
        level = "SL"
        dist = column_name[3:6]
    elif column_name[0:3] == "GSU" or column_name[0:3] == "PSU":
        level = "SU"
        dist = column_name[3:5]
    elif column_name[0:3] == "GCO"  or column_name[0:3] == "PCO":
        level = "CON"
        dist = column_name[4:6]
    else:
        print(column_name)
        raise ValueError
    return level,dist

def contains_sldl(dist_list):
    for dist_tuple in dist_list:
        if dist_tuple[0] == "SL":
            return dist_tuple[1]
        
def contains_cong(dist_list):
    for dist_tuple in dist_list:
        if dist_tuple[0] == "CON":
            return dist_tuple[1]
        
def contains_sldu(dist_list):
    for dist_tuple in dist_list:
        if dist_tuple[0] == "SU":
            return dist_tuple[1]

precinct_mapping_dict = {}
split_precincts_list = {}
for index,row in final_join.iterrows():
    precinct_list = []
    for contest in races:
        if(row[contest]!=0) and ("GSL" in contest or "GCO" in contest or "GSU" in contest or "PCO" in contest or "PSU" in contest or "PSL" in contest):
            precinct_info = get_level_dist(contest)
            if precinct_info not in precinct_list:
                precinct_list.append(get_level_dist(contest))
    is_split = is_split_precinct(precinct_list)
    if (is_split):
        split_precincts_list[row["UNIQUE_ID"]]=is_split
    precinct_mapping_dict[row["UNIQUE_ID"]]=precinct_list
    
cong_check_list = {i:contains_cong(precinct_mapping_dict[i]) for i in precinct_mapping_dict.keys()}

In [129]:
def return_splits(split_dict, level):
    for val in split_dict.keys():
        if level in val:
            return split_dict[level]

def create_splits_dict(level):
    level_splits_dict = {i:return_splits(split_precincts_list[i], level) for i in split_precincts_list.keys() if return_splits(split_precincts_list[i], level) != None }
    return level_splits_dict

In [130]:
cong_splits_dict = create_splits_dict('CON')


In [131]:
## Clean up N/A District Assignments


def clean_na_dist_assignments(elections_gdf, district_gdf, unique_ID_col, elections_gdf_dist_ID, district_gdf_ID, ):
    
    if elections_gdf[elections_gdf[elections_gdf_dist_ID].isna()].shape[0]==0:
        return elections_gdf
    
    else:
        print("Fixing ",elections_gdf[elections_gdf[elections_gdf_dist_ID].isna()].shape[0], "precincts")
    
    original_crs = elections_gdf.crs
    elections_gdf = elections_gdf.to_crs(3857)
    
    district_gdf = district_gdf.to_crs(3857)
    
    dist_clean = gp.overlay(elections_gdf[elections_gdf[elections_gdf_dist_ID].isna()], district_gdf, how = "intersection")

    dist_clean['area'] = dist_clean.area

    na_assignment_dict = {}

    for val in dist_clean[unique_ID_col].unique():

        assignment = dist_clean.loc[dist_clean[unique_ID_col] == val].nlargest(1, 'area')[district_gdf_ID].values[0]
        na_assignment_dict[val] = assignment

    elections_gdf[elections_gdf_dist_ID] = elections_gdf[unique_ID_col].map(na_assignment_dict).fillna(elections_gdf[elections_gdf_dist_ID])    

    elections_gdf = elections_gdf.to_crs(original_crs)
    
    return elections_gdf

In [132]:
CONG_PATH = "/Users/peterhorton/Documents/RDH/raw_data/cong/national_cong119_boundary/national_cong119_boundary/national_cong119_boundary.shp"

# Load in CONG and SLDU shapefiles
la_cong_districts = gp.read_file(CONG_PATH)

la_cong_districts = la_cong_districts.to_crs(final_join.crs)

la_cong_districts = la_cong_districts[la_cong_districts["STATE"]=="LA"]

final_join["CONG_DIST"] = final_join["UNIQUE_ID"].map(cong_check_list)



In [133]:
# # Clean na district assignments
final_join = clean_na_dist_assignments(final_join, la_cong_districts, "UNIQUE_ID", "CONG_DIST", "DISTRICT")

Fixing  17 precincts


/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_9433/3579320801.py:17: UserWarning: `keep_geom_type=True` in overlay resulted in 110 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  dist_clean = gp.overlay(elections_gdf[elections_gdf[elections_gdf_dist_ID].isna()], district_gdf, how = "intersection")


In [149]:
cong_splits_dict

{'Avoyelles-:-4-2A': ['05', '06']}

In [134]:
cong_columns = [i for i in list(final_join.columns) if "GCON" in i]

In [135]:
la_cong_districts.rename(columns = {"CONG_DIST":"DISTRICT"}, inplace = True)
final_join_unsplit = final_join.copy(deep = True)
final_join, cong_diff = pdv.district_splits_comb("CONG", list(cong_splits_dict.keys()), final_join, la_cong_districts, "UNIQUE_ID", "DISTRICT", cong_columns, "CONG_DIST")


/Users/peterhorton/Documents/RDH/pber_local/LA_2024/general/pber_functions_v1.py:440: UserWarning: `keep_geom_type=True` in overlay resulted in 85 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  test_join = gp.overlay(need_splits, district_gdf, how="intersection")


In [136]:

final_join["CONG_DIST"] = final_join["CONG_DIST"].astype(str).str.zfill(2)

la_cong_districts.rename(columns = {"DISTRICT":"CONG_DIST"}, inplace = True)

filtered_cong_results = final_join[~final_join["CONG_DIST"].isna()].dissolve("CONG_DIST")
filtered_cong_results.reset_index(inplace = True, drop = False)

la_cong_districts["CONG_DIST"] = la_cong_districts["CONG_DIST"].astype(str).str.zfill(2)



# Check implied SLDU districts based off of assignments against the actual ones
pdv.compare_geometries(filtered_cong_results, la_cong_districts ,"Election Results", "Census", "CONG_DIST","Districts",area_threshold=.01)










Checking 6 Districts for differences of greater than 0.01 km^2



/Users/peterhorton/Documents/RDH/pber_local/LA_2024/general/pber_functions_v1.py:366: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  area = float(diff.area/10e6)



Scroll down to see plots of any differences

Of the 6 Districts:

4 Districts w/ a difference of 0 km^2
2 Districts w/ a difference between 0 and .1 km^2
0 Districts w/ a difference between .1 and .5 km^2
0 Districts w/ a difference between .5 and 1 km^2
0 Districts w/ a difference between 1 and 2 km^2
0 Districts w/ a difference between 2 and 5 km^2
0 Districts w/ a difference greater than 5 km^2


In [137]:
final_join.rename(columns = {"COUNTYFP20":"COUNTYFP"}, inplace = True)

## Check Implied Parish Assignments

COUNTY_PATH = "/Users/peterhorton/Documents/RDH/raw_data/census/2020_TIGER_CNTY/la_cnty_2020_bound/la_cnty_2020_bound.shp"

la_counties = gp.read_file(COUNTY_PATH)
la_counties.rename(columns = {"COUNTYFP20":"COUNTYFP"}, inplace = True)
grouped_counties = final_join.dissolve("COUNTYFP")
grouped_counties.reset_index(inplace = True, drop = False)

pdv.compare_geometries(la_counties, grouped_counties, "Census", "File", "COUNTYFP", "County")


Checking 64 County for differences of greater than 0.1 km^2



/Users/peterhorton/Documents/RDH/pber_local/LA_2024/general/pber_functions_v1.py:366: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  area = float(diff.area/10e6)



Scroll down to see plots of any differences

Of the 64 County:

48 County w/ a difference of 0 km^2
16 County w/ a difference between 0 and .1 km^2
0 County w/ a difference between .1 and .5 km^2
0 County w/ a difference between .5 and 1 km^2
0 County w/ a difference between 1 and 2 km^2
0 County w/ a difference between 2 and 5 km^2
0 County w/ a difference greater than 5 km^2


In [139]:
final_join.sort_values("UNIQUE_ID", inplace = True)
final_join_unsplit.sort_values("UNIQUE_ID", inplace = True)

In [140]:
final_join = final_join[["UNIQUE_ID","COUNTYFP","Parish","CONG_DIST","NAME20","GEOID20"]+cong_columns+["geometry"]]
final_join_unsplit = final_join_unsplit[["UNIQUE_ID","COUNTYFP","Parish","NAME20","GEOID20"]+races+["geometry"]]

In [141]:
for race in races:
    final_join_unsplit[race] = final_join_unsplit[race].astype(int)
    
for race in cong_columns:
    final_join[race] = final_join[race].astype(int)

In [144]:
if not os.path.exists("./la_2024_gen_prec"):
    os.mkdir("./la_2024_gen_prec/")
    
if not os.path.exists("./la_2024_gen_prec/la_2024_gen_cong_prec"):
    os.mkdir("./la_2024_gen_prec/la_2024_gen_cong_prec")
if not os.path.exists("./la_2024_gen_prec/la_2024_gen_all_prec"):
    os.mkdir("./la_2024_gen_prec/la_2024_gen_all_prec")
    
        
final_join.to_file("./la_2024_gen_prec/la_2024_gen_cong_prec/la_2024_gen_cong_prec.shp")
final_join_unsplit.to_file("./la_2024_gen_prec/la_2024_gen_all_prec/la_2024_gen_all_prec.shp")



In [150]:
import shutil

In [151]:
shutil.make_archive("./la_2024_gen_prec","zip","./la_2024_gen_prec")

'/Users/peterhorton/Documents/RDH/pber_local/LA_2024/general/la_2024_gen_prec.zip'